# Triagem de pedidos de manutenção — Projeto de IA

**Grupo:** _(número)_

| Nome | Número |
|---|---|
| _(preencher)_ | |
| _(preencher)_ | |
| _(preencher)_ | |

Estrutura esperada: este notebook ao lado da pasta `dados/` com `treino.csv`, `validacao.csv`, `teste.csv` e `rotulos_teste.csv`.

## 1. Introdução — o problema como agente (PEAS)

| PEAS | Neste problema |
|---|---|
| **Desempenho** | _(TODO: custo médio com FN = 4 × FP, recall, taxa de sinalização…)_ |
| **Ambiente** | _(TODO)_ |
| **Atuadores** | _(TODO)_ |
| **Sensores** | _(TODO)_ |

## 2. Configuração, constantes e leitura dos dados

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score)
import sklearn

print("pandas", pd.__version__, "| numpy", np.__version__, "| sklearn", sklearn.__version__)

In [ ]:
# Constantes do protocolo: mudar aqui e voltar a correr tudo.
SEED = 42                      # fixada antes de ver qualquer resultado
CUSTO_FN = 4                   # um falso negativo custa 4 vezes um falso positivo
CUSTO_FP = 1
LIMIARES = [0.20, 0.35, 0.50, 0.65, 0.80]

NUM_COLS = ["utilizadores_afetados", "horas_desde_deteccao", "reincidencias_30d", "idade_anos"]
CAT_COLS = ["equipamento", "contexto", "interrupcao_total", "alternativa_disponivel",
            "sinal_seguranca", "periodo"]
ALVO = "urgente"
ID = "id_pedido"

assert ALVO not in NUM_COLS + CAT_COLS and ID not in NUM_COLS + CAT_COLS
assert len(NUM_COLS + CAT_COLS) == 10

In [ ]:
tr = pd.read_csv("dados/treino.csv")
va = pd.read_csv("dados/validacao.csv")
te = pd.read_csv("dados/teste.csv")                 # NOTA: não traz a coluna 'urgente'
rotulos_te = pd.read_csv("dados/rotulos_teste.csv") # rótulos do teste: só usados na secção final

# Verificações estruturais (permitidas também no teste; não exploram 'urgente')
for nome, df in [("treino", tr), ("validacao", va), ("teste", te), ("rotulos_teste", rotulos_te)]:
    print(f"{nome:14s} {df.shape}  ids únicos: {df[ID].is_unique}")

assert not set(tr[ID]) & set(va[ID])
assert not set(tr[ID]) & set(te[ID])
assert not set(va[ID]) & set(te[ID])
assert set(te[ID]) == set(rotulos_te[ID])

X_tr, y_tr = tr[NUM_COLS + CAT_COLS], tr[ALVO]
X_va, y_va = va[NUM_COLS + CAT_COLS], va[ALVO]
X_te = te[NUM_COLS + CAT_COLS]

## 3. Auditoria (só treino)

TODO:
- tabela com coluna, tipo, unidade, domínio esperado, nº e % de valores em falta;
- ids repetidos, linhas duplicadas, categorias inesperadas, valores fora do domínio, contagens não inteiras, extremos (IQR);
- registo de problemas: Problema · Evidência · Decisão · Linhas afetadas · Efeito (incluir as decisões de **não** alterar).

In [ ]:
# TODO: tabela de auditoria

## 4. Experiência de imputação (numa cópia)

TODO: escolher uma coluna numérica com omissões, preencher numa cópia com a mediana do treino e comparar antes/depois
(histograma + caixa de bigodes, mesmos intervalos e escalas). O `tr` original não pode ser alterado.

In [ ]:
# TODO

## 5. Figuras obrigatórias

Cada figura: pergunta, porquê este gráfico, interpretação, limitação, denominador das percentagens.

### Figura 1 — % de valores em falta por atributo

**Pergunta:** …  
**Porquê este gráfico:** …  
**Interpretação:** …  
**Limitação:** …

In [ ]:
# TODO: figura 1

### Figura 2 — Pedidos por equipamento ou contexto (contagens vs proporções)

**Pergunta:** …  
**Porquê este gráfico:** …  
**Interpretação:** …  
**Limitação:** …

In [ ]:
# TODO: figura 2

### Figura 3 — Distribuição de um atributo numérico antes/depois da imputação (painéis)

**Pergunta:** …  
**Porquê este gráfico:** …  
**Interpretação:** …  
**Limitação:** …

In [ ]:
# TODO: figura 3

### Figura 4 — Atributo numérico por equipamento ou contexto

**Pergunta:** …  
**Porquê este gráfico:** …  
**Interpretação:** …  
**Limitação:** …

In [ ]:
# TODO: figura 4

### Figura 5 — Proporção de urgentes por categoria, com n total

**Pergunta:** …  
**Porquê este gráfico:** …  
**Interpretação:** …  
**Limitação:** …

In [ ]:
# TODO: figura 5

## 6. Síntese da primeira fase (300–500 palavras)

_TODO: três observações concretas e duas questões a explorar na modelação._

## 7. Preparação dos dados

Numéricas: mediana (+ escala só para o MLP). Categóricas: categoria explícita `em_falta` + one-hot com
`handle_unknown="ignore"`. Tudo dentro de um Pipeline, para o ajuste ser feito só no treino.

_TODO: justificar cada escolha por palavras vossas._

In [ ]:
def make_prep(escalar):
    num_steps = [("imputar", SimpleImputer(strategy="median"))]
    if escalar:
        num_steps.append(("escalar", StandardScaler()))
    cat_steps = [
        ("imputar", SimpleImputer(strategy="constant", fill_value="em_falta")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
    return ColumnTransformer([
        ("num", Pipeline(num_steps), NUM_COLS),
        ("cat", Pipeline(cat_steps), CAT_COLS),
    ])

def make_pipe(modelo, escalar):
    return Pipeline([("prep", make_prep(escalar)), ("modelo", clone(modelo))])

In [ ]:
# Verificação: a mediana aprendida tem de ser a do treino
prep = make_prep(escalar=True).fit(X_tr)
print("forma na validação:", prep.transform(X_va).shape)
print("medianas aprendidas:", prep.named_transformers_["num"]["imputar"].statistics_)
print("medianas do treino: ", X_tr[NUM_COLS].median().values)

## 8. Métricas

Taxa de sinalização e custo médio implementados à mão; verificados com o exemplo do Anexo A.

In [ ]:
def custo_medio(y, y_pred, c_fn=CUSTO_FN, c_fp=CUSTO_FP):
    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
    return (c_fn * fn + c_fp * fp) / len(y)

def taxa_sinalizacao(y_pred):
    return np.mean(np.asarray(y_pred) == 1)

def metricas(y, y_pred):
    tn, fp, fn, tp = confusion_matrix(y, y_pred, labels=[0, 1]).ravel()
    return {
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "taxa_sinalizacao": taxa_sinalizacao(y_pred),
        "custo_medio": custo_medio(y, y_pred),
        "sem_sinalizados": (tp + fp) == 0,   # assinala quando precision vem de zero_division
    }

# Verificação com o Anexo A: TN=60, FP=15, FN=5, TP=20
y_ex = np.array([0] * 75 + [1] * 25)
p_ex = np.array([0] * 60 + [1] * 15 + [0] * 5 + [1] * 20)
m = metricas(y_ex, p_ex)
assert np.isclose(m["accuracy"], 0.80) and np.isclose(m["recall"], 0.80)
assert np.isclose(m["precision"], 20 / 35) and np.isclose(m["f1"], 2 / 3)
assert np.isclose(m["taxa_sinalizacao"], 0.35) and np.isclose(m["custo_medio"], 0.35)
print("Anexo A confere.")

## 9. Referências simples (validação)

1. Classe maioritária do **treino**.
2. Todos urgentes.
3. Regra do grupo: _TODO — escrever a regra e como trata omissões ANTES de a correr._

In [ ]:
# TODO: calcular metricas(...) na validação para as três referências

## 10. Configurações e tabela de experiências

Ordem de preferência (simplicidade) registada **antes** de treinar: _TODO justificar._

In [ ]:
# Família -> rank de preferência para desempates (árvore, depois RF, depois MLP)
RANK_FAMILIA = {"arvore": 0, "rf": 1, "mlp": 2}

CONFIGS = {
    # id: (família, modelo, escalar, rank de complexidade dentro da família)
    "DT-A":  ("arvore", DecisionTreeClassifier(max_depth=3, min_samples_leaf=20, random_state=SEED), False, 0),
    "DT-B":  ("arvore", DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=SEED), False, 1),
    "RF-A":  ("rf", RandomForestClassifier(n_estimators=100, min_samples_leaf=10, random_state=SEED), False, 0),
    "RF-B":  ("rf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=SEED), False, 1),
    "MLP-A": ("mlp", MLPClassifier(hidden_layer_sizes=(8,), max_iter=300, random_state=SEED), True, 0),
    "MLP-B": ("mlp", MLPClassifier(hidden_layer_sizes=(16, 8), alpha=1e-3, max_iter=1000, random_state=SEED), True, 1),
}
assert 6 <= len(CONFIGS) <= 9

In [ ]:
# TODO: para cada configuração
#   - make_pipe, fit em X_tr / y_tr (capturar ConvergenceWarning)
#   - i1 = list(pipe.classes_).index(1); probabilidades de treino e validação
#   - métricas a 0,50 no treino e na validação -> linha da tabela de experiências
#   - guardar as probabilidades da validação em probas_val[exp_id]
probas_val = {}

## 11. Experiência controlada

**Alterámos…** **Esperávamos…** **porque…** **Observámos…** na validação. **Concluímos…**

## 12. Comparação de limiares e escolha do par configuração + limiar

In [ ]:
# TODO: grelha configuração x LIMIARES (proba >= limiar), métricas na validação,
#       ordenar por: custo ↑, F1 ↓, família, complexidade, |limiar-0.5| ↑, limiar ↑ (sem arredondar)

## 13. Análise de erros na validação (3 FP e 3 FN, modelo treinado só no treino)

In [ ]:
# TODO

## 14. Decisão final (escrita antes de olhar para o teste)

- Atributos: …
- Preparação: …
- Família e hiperparâmetros: …
- Semente: …
- Limiar: …
- Justificação (valores da validação): …

## 15. Ajuste final em treino + validação e avaliação no teste

In [ ]:
# TODO:
# X_tv = pd.concat([X_tr, X_va]); y_tv = pd.concat([y_tr, y_va])
# final = make_pipe(<modelo escolhido>, <escalar>).fit(X_tv, y_tv)   # make_pipe já faz clone
# te_rot = te.merge(rotulos_te, on=ID, validate="one_to_one")       # rótulos só agora
# probabilidades -> classes com o limiar fixo -> metricas(te_rot[ALVO], ...)

## 16. Limitações, contribuições e declaração de utilização de IA

_TODO_